In [34]:
import pandas as pd
import re
import string
from io import StringIO
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# ----------------------------
# DOWNLOAD DATA & NLTK SETUP
# ----------------------------
try:
    nltk.download('stopwords', quiet=True)
    stop_words_id = set(stopwords.words('indonesian'))
    stop_words_en = set(stopwords.words('english'))
    stop_words = stop_words_id.union(stop_words_en)
except Exception as e:
    print("NLTK stopwords gagal di-download. Menggunakan manual list.")
    stop_words = {
        'dan', 'di', 'ke', 'yang', 'adalah', 'ini', 'itu', 'dari', 'untuk', 'pada',
        'dengan', 'ya', 'atau', 'jadi', 'tapi', 'juga', 'kita', 'saya', 'dia', 'mereka',
        'kamu', 'kalian', 'kami', 'sih', 'loh', 'deh', 'dong', 'nih', 'banget', 'aja',
        'makanya', 'gitu', 'gini', 'doang', 'tau', 'tahu', 'pengen', 'mau', 'bikin'
    }

# ----------------------------------------
# DICTIONARY KOREKSI SLANG / TYPO (LENGKAP)
# ----------------------------------------
typo_corrections = {
    # Umum
    'tau': 'tahu', 'ga': 'tidak', 'ngirip': 'menginap', 'nggak': 'tidak',
    'gak': 'tidak', 'gila': 'sangat', 'gue': 'saya', 'guwa': 'saya', 'gua': 'saya',
    'lu': 'kamu', 'lo': 'kamu', 'denger': 'mendengar', 'denger-denger': 'mendengar-dengar',
    'ngerti': 'mengerti', 'ngambil': 'mengambil', 'nyetir': 'menyetir', 'ngejimpai': 'pergi ke gym',
    'pengen': 'ingin', 'mau': 'ingin', 'bikin': 'membuat', 'bkin': 'membuat',
    'udah': 'sudah', 'abis': 'habis', 'abisin': 'habiskan', 'abisin': 'habiskan',
    'kalo': 'kalau', 'klo': 'kalau', 'kpn': 'kapan', 'dr': 'dari', 'bs': 'bisa',
    'skrg': 'sekarang', 'kmrn': 'kemarin', 'blm': 'belum', 'bgt': 'banget',
    'sm': 'sama', 'yg': 'yang', 'dgn': 'dengan', 'jd': 'jadi', 'tp': 'tapi',
    'sy': 'saya', 'km': 'kamu', 'brp': 'berapa', 'aj': 'saja', 'aj': 'saja',
    'temen': 'teman', 'temen-temen': 'teman-teman', 'org': 'orang', 'dpt': 'dapat',
    'krn': 'karena', 'krna': 'karena', 'cm': 'cuma', 'cmn': 'cuma',
    'knp': 'kenapa', 'bsk': 'besok', 'td': 'tadi', 'dtg': 'datang', 'dkt': 'dekat',
    'lg': 'lagi', 'lgsg': 'langsung', 'pdhl': 'padahal', 'dri': 'dari',
    'kmbli': 'kembali', 'kmna': 'kemana', 'kmrn': 'kemarin', 'mslh': 'masalah',
    'dprnahkn': 'dipernahkan', 'bwt': 'buat', 'biar': 'agar', 'krja': 'kerja',
    'lgsung': 'langsung', 'msh': 'masih', 'slh': 'salah', 'byr': 'bayar',
    'plg': 'paling', 'bkn': 'bukan', 'dkt': 'dekat', 'dlm': 'dalam',
    'smpe': 'sampai', 'smpai': 'sampai', 'dunia': 'world', 'game': 'permainan',
    'top up': 'isi ulang', 'laptop': 'komputer jinjing', 'monitor': 'layar',
    'gamepad': 'kontroler permainan', 'smartwatch': 'jam pintar', 'strava': 'aplikasi olahraga',

    # Produk/Brand
    'jqj8rd': 'jaecoo j8', 'skomadi': 'scomadi', 'miux': 'mu-x', 'fronks': 'fronx',
    'gias': 'GIIAS', 'carnet': 'carnet', 'warn': 'warna', 'zen tableware': 'zen tableware',
    'kunta residense': 'khunta residence', 'hongsong laos': 'rumah tradisional laos',
    'hong song laos': 'rumah tradisional laos', 'luang say residence': 'luang say residence',
    'saint gobain': 'saint gobain', 'reak native': 'react native', 'steviat': 'stevia',
    'koffie': 'kopi', 'bandics': 'rem', 'overlaid': 'overland', 'destinator': 'destinator',
    'compost it': 'composter', 'kemperven': 'campervan', 'tek-tek': 'tiktok',
    'tulkit': 'perlengkapan', 'jurni': 'perjalanan', 'jurniannya': 'jadinya',
    'permomenarik': 'promo menarik', 'sefa': 'sefa', 'zink': 'zing', 'zing': 'zing',
    'gladrags': 'gladrags', 'wada': 'WADA', 'gtd': 'GTD', 'dedy lawab': 'Dedy Lawab',
    'nazifah': 'Nazifah', 'mk pratikno': 'Menteri Pratikno', 'lpd': 'LPD', 'pm': 'PM',
    'elon technik': 'Elon Teknik', 'el ton tenggu': 'El Ton Tenggu', 'john': 'John',
    'alex andro': 'Alex Andro', 'sada': 'Sada', 'naya': 'Naya', 'fibra struck': 'Fibre Struck',
    'scomadi': 'Scomadi', 'vs code': 'Visual Studio Code', 'gt top up': 'GTD Top Up',
    'legion pro 7i': 'Legion Pro 7i', 'rtx 5090': 'RTX 5090', 'intel core ultra 9': 'Intel Core Ultra 9',
    'lenovo': 'Lenovo', 'cyberpang': 'Cyberpunk', 'black mids mookong': 'Black Myth Wukong',
}

# ----------------------------
# FUNGSI CLEANING YANG DIPERBAIKI
# ----------------------------
def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)
    
    # Hapus emoji dan karakter non-ASCII (e.g., ëžŒëª¨ì‚¬)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    
    # Koreksi typo berdasarkan dictionary
    for slang, formal in typo_corrections.items():
        pattern = r'\b' + re.escape(slang) + r'\b'
        text = re.sub(pattern, formal, text, flags=re.IGNORECASE)

    # Case folding
    text = text.lower()

    # Hapus URL
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)

    # Hapus email
    text = re.sub(r'\S+@\S+', '', text)

    # Hapus angka
    text = re.sub(r'\d+', '', text)

    # Hapus tanda baca (kecuali . ! ? , untuk menjaga intonasi)
    keep_punct = ['!', '?', '.', ',']
    translator = str.maketrans('', '', ''.join([c for c in string.punctuation if c not in keep_punct]))
    text = text.translate(translator)

    # Kurangi huruf berulang >2 kali (e.g., waaaaaaah -> waah)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # Normalisasi spasi
    text = re.sub(r'\s+', ' ', text).strip()

    # Hapus stopwords dan kata sangat pendek
    words = [word for word in text.split() if word not in stop_words and len(word) > 2]
    text = ' '.join(words)

    return text.strip()

# ----------------------------
# LOAD DATA
# ----------------------------
file_path = "/kaggle/input/text-bdc/hasil_preprocessing.csv"  # Pastikan file ada di direktori kerja
try:
    df = pd.read_csv(file_path)
    print(f"File loaded successfully. Shape: {df.shape}")
except FileNotFoundError:
    print(f"File not found at {file_path}. Using sample data.")
    sample_csv = """id,emotion,text_transcript
317,Surprise,"Ongkutan denger-denger kata lu kalau nutuk laptop lama-lama sambil menitoran bukan berusaha klare laptopnya."
643,Surprise,"seneng banget rasanya aku diundang ke universitas sujai kusuma surabaya untuk ngajarin operasi ikan di valputus kedopteran"
264,Joy,"Hello, semuanya! Aku Nazifah dari Kota Bandung. Hello, dari Gila Ro. Aku dari Mokasi dan aku dari sini dari Jakarta."
802,Anger,"ini dia yang katanya text editor pengganti VS Code yang udah build in AI nya gratis"
497,Proud,"Bersama mengkopi Mk. Pratikno, kemenpora bersama LPD, PM meluncurkan biasiswa kaolah ragaan"
"""
    df = pd.read_csv(StringIO(sample_csv))

# Validasi kolom
if 'text_transcript' not in df.columns:
    text_col = df.select_dtypes(include=['object']).columns[0]
    df = df.rename(columns={text_col: 'text_transcript'})
    print(f"Renamed column to 'text_transcript'.")

# Drop baris kosong
df.dropna(subset=['text_transcript'], inplace=True)

# Encode label
le = LabelEncoder()
df['label'] = le.fit_transform(df['emotion'])
print(f"Classes: {le.classes_.tolist()}")

# ----------------------------
# SPLIT DATA
# ----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    df['text_transcript'],
    df['label'],
    test_size=0.27,
    stratify=df['label'],
)

# ----------------------------
# PREPROCESSING
# ----------------------------
train_df = pd.DataFrame({'text': X_train, 'label': y_train})
test_df = pd.DataFrame({'text': X_test, 'label': y_test})

train_df['cleaned_text'] = train_df['text'].apply(clean_text)
test_df['cleaned_text'] = test_df['text'].apply(clean_text)

# Buang jika cleaned_text kosong
train_df = train_df[train_df['cleaned_text'] != ""].reset_index(drop=True)
test_df = test_df[test_df['cleaned_text'] != ""].reset_index(drop=True)

print("\n=== CLEANING RESULTS ===")
print(f"Train set: {len(train_df)} samples")
print(f"Test set: {len(test_df)} samples")
print("\nSample cleaned text (train):")
print(train_df[['cleaned_text']].head(3))

# ----------------------------
# TF-IDF VECTORIZATION
# ----------------------------
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000, min_df=2, max_df=0.95)
X_train_vec = vectorizer.fit_transform(train_df['cleaned_text'])
X_test_vec = vectorizer.transform(test_df['cleaned_text'])

print(f"\nTF-IDF Train shape: {X_train_vec.shape}")
print(f"TF-IDF Test shape: {X_test_vec.shape}")

# ----------------------------
# MODEL TRAINING
# ----------------------------
model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
model.fit(X_train_vec, train_df['label'])

# Prediksi
y_pred = model.predict(X_test_vec)

# ----------------------------
# EVALUASI
# ----------------------------
print("\n=== MODEL EVALUATION ===")
print("Confusion Matrix:")
print(confusion_matrix(test_df['label'], y_pred))
print("\nClassification Report:")
print(classification_report(test_df['label'], y_pred, target_names=le.classes_))

# ----------------------------
# SIMPAN HASIL
# ----------------------------
train_df.to_csv('train_data_cleaned.csv', index=False)
test_df.to_csv('test_data_cleaned.csv', index=False)
print("\nData saved to 'train_data_cleaned.csv' and 'test_data_cleaned.csv'.")

# Opsional: Simpan model dan vectorizer
import joblib
joblib.dump(model, 'logreg_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
joblib.dump(le, 'label_encoder.pkl')
print("Model, vectorizer, and encoder saved.")

File loaded successfully. Shape: (738, 14)
Classes: ['Anger', 'Fear', 'Joy', 'Neutral', 'Proud', 'Sadness', 'Surprise', 'Trust']

=== CLEANING RESULTS ===
Train set: 538 samples
Test set: 200 samples

Sample cleaned text (train):
                                        cleaned_text
0  halo semuanya, mobil special hyundai ioniq bar...
1  ladian otor kaki rumah ala dyunitonres. wqot b...
2  susah banget, susah banget susah kan? susah ng...

TF-IDF Train shape: (538, 5000)
TF-IDF Test shape: (200, 5000)

=== MODEL EVALUATION ===
Confusion Matrix:
[[ 3  0  0  0  1  0  5  1]
 [ 0  1  0  0  1  0  2  0]
 [ 0  0  2  0  3  0  5  4]
 [ 0  1  0  0  1  0  0  0]
 [ 2  0  3  0 15  0 14  5]
 [ 0  0  0  0  0  1  2  1]
 [ 4  0  4  0 15  1 36 23]
 [ 0  0  5  0 14  0 10 15]]

Classification Report:
              precision    recall  f1-score   support

       Anger       0.33      0.30      0.32        10
        Fear       0.50      0.25      0.33         4
         Joy       0.14      0.14      0.14   

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Model, vectorizer, and encoder saved.


In [35]:
import pandas as pd
import joblib
import re
import string
import nltk
from nltk.corpus import stopwords
import numpy as np

# ----------------------------
# DOWNLOAD DATA & NLTK SETUP
# ----------------------------
try:
    nltk.download('stopwords', quiet=True)
    stop_words_id = set(stopwords.words('indonesian'))
    stop_words_en = set(stopwords.words('english'))
    stop_words = stop_words_id.union(stop_words_en)
except Exception as e:
    print("NLTK stopwords gagal di-download. Menggunakan manual list.")
    stop_words = {
        'dan', 'di', 'ke', 'yang', 'adalah', 'ini', 'itu', 'dari', 'untuk', 'pada',
        'dengan', 'ya', 'atau', 'jadi', 'tapi', 'juga', 'kita', 'saya', 'dia', 'mereka',
        'kamu', 'kalian', 'kami', 'sih', 'loh', 'deh', 'dong', 'nih', 'banget', 'aja',
        'makanya', 'gitu', 'gini', 'doang', 'tau', 'tahu', 'pengen', 'mau', 'bikin'
    }

# ----------------------------------------
# DICTIONARY KOREKSI SLANG / TYPO
# ----------------------------------------
typo_corrections = {
    'tau': 'tahu', 'ga': 'tidak', 'ngirip': 'menginap', 'nggak': 'tidak',
    'gak': 'tidak', 'gila': 'sangat', 'gue': 'saya', 'guwa': 'saya', 'gua': 'saya',
    'lu': 'kamu', 'lo': 'kamu', 'denger': 'mendengar', 'denger-denger': 'mendengar-dengar',
    'ngerti': 'mengerti', 'ngambil': 'mengambil', 'nyetir': 'menyetir', 'ngejimpai': 'pergi ke gym',
    'pengen': 'ingin', 'mau': 'ingin', 'bikin': 'membuat', 'bkin': 'membuat',
    'udah': 'sudah', 'abis': 'habis', 'abisin': 'habiskan', 'abisin': 'habiskan',
    'kalo': 'kalau', 'klo': 'kalau', 'kpn': 'kapan', 'dr': 'dari', 'bs': 'bisa',
    'skrg': 'sekarang', 'kmrn': 'kemarin', 'blm': 'belum', 'bgt': 'banget',
    'sm': 'sama', 'yg': 'yang', 'dgn': 'dengan', 'jd': 'jadi', 'tp': 'tapi',
    'sy': 'saya', 'km': 'kamu', 'brp': 'berapa', 'aj': 'saja', 'aj': 'saja',
    'temen': 'teman', 'temen-temen': 'taman-taman', 'org': 'orang', 'dpt': 'dapat',
    'krn': 'karena', 'krna': 'karena', 'cm': 'cuma', 'cmn': 'cuma',
    'knp': 'kenapa', 'bsk': 'besok', 'td': 'tadi', 'dtg': 'datang', 'dkt': 'dekat',
    'lg': 'lagi', 'lgsg': 'langsung', 'pdhl': 'padahal', 'dri': 'dari',
    'kmbli': 'kembali', 'kmna': 'kemana', 'kmrn': 'kemarin', 'mslh': 'masalah',
    'dprnahkn': 'dipernahkan', 'bwt': 'buat', 'biar': 'agar', 'krja': 'kerja',
    'lgsung': 'langsung', 'msh': 'masih', 'slh': 'salah', 'byr': 'bayar',
    'plg': 'paling', 'bkn': 'bukan', 'dkt': 'dekat', 'dlm': 'dalam',
    'smpe': 'sampai', 'smpai': 'sampai', 'dunia': 'world', 'game': 'permainan',
    'top up': 'isi ulang', 'laptop': 'komputer jinjing', 'monitor': 'layar',
    'gamepad': 'kontroler permainan', 'smartwatch': 'jam pintar', 'strava': 'aplikasi olahraga',
    'jqj8rd': 'jaecoo j8', 'skomadi': 'scomadi', 'miux': 'mu-x', 'fronks': 'fronx',
    'gias': 'GIIAS', 'carnet': 'carnet', 'warn': 'warna', 'zen tableware': 'zen tableware',
    'kunta residense': 'khunta residence', 'hongsong laos': 'rumah tradisional laos',
    'hong song laos': 'rumah tradisional laos', 'luang say residence': 'luang say residence',
    'saint gobain': 'saint gobain', 'reak native': 'react native', 'steviat': 'stevia',
    'koffie': 'kopi', 'bandics': 'rem', 'overlaid': 'overland', 'destinator': 'destinator',
    'compost it': 'composter', 'kemperven': 'campervan', 'tek-tek': 'tiktok',
    'tulkit': 'perlengkapan', 'jurni': 'perjalanan', 'jurniannya': 'jadinya',
    'permomenarik': 'promo menarik', 'sefa': 'sefa', 'zink': 'zing', 'zing': 'zing',
    'gladrags': 'gladrags', 'wada': 'WADA', 'gtd': 'GTD', 'dedy lawab': 'Dedy Lawab',
    'nazifah': 'Nazifah', 'mk pratikno': 'Menteri Pratikno', 'lpd': 'LPD', 'pm': 'PM',
    'elon technik': 'Elon Teknik', 'el ton tenggu': 'El Ton Tenggu', 'john': 'John',
    'alex andro': 'Alex Andro', 'sada': 'Sada', 'naya': 'Naya', 'fibra struck': 'Fibre Struck',
    'scomadi': 'Scomadi', 'vs code': 'Visual Studio Code', 'gt top up': 'GTD Top Up',
    'legion pro 7i': 'Legion Pro 7i', 'rtx 5090': 'RTX 5090', 'intel core ultra 9': 'Intel Core Ultra 9',
    'lenovo': 'Lenovo', 'cyberpang': 'Cyberpunk', 'black mids mookong': 'Black Myth Wukong',
}

# ----------------------------
# FUNGSI CLEANING
# ----------------------------
def clean_text(text):
    if pd.isna(text) or text == "":
        return ""

    text = str(text)
    
    # Hapus emoji dan karakter non-ASCII
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    
    # Koreksi typo berdasarkan dictionary
    for slang, formal in typo_corrections.items():
        pattern = r'\b' + re.escape(slang) + r'\b'
        text = re.sub(pattern, formal, text, flags=re.IGNORECASE)

    # Case folding
    text = text.lower()

    # Hapus URL
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)

    # Hapus email
    text = re.sub(r'\S+@\S+', '', text)

    # Hapus angka
    text = re.sub(r'\d+', '', text)

    # Hapus tanda baca (kecuali . ! ? ,)
    keep_punct = ['!', '?', '.', ',']
    translator = str.maketrans('', '', ''.join([c for c in string.punctuation if c not in keep_punct]))
    text = text.translate(translator)

    # Kurangi huruf berulang >2 kali
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # Normalisasi spasi
    text = re.sub(r'\s+', ' ', text).strip()

    # Hapus stopwords dan kata pendek
    words = [word for word in text.split() if word not in stop_words and len(word) > 2]
    text = ' '.join(words)

    return text.strip()

# ----------------------------
# LOAD MODEL DAN ENCODER
# ----------------------------
try:
    model = joblib.load('logreg_model.pkl')
    vectorizer = joblib.load('tfidf_vectorizer.pkl')
    le = joblib.load('label_encoder.pkl')
    print("Model, vectorizer, and encoder loaded successfully.")
except FileNotFoundError:
    print("Error: Model, vectorizer, or encoder file not found. Please ensure 'logreg_model.pkl', 'tfidf_vectorizer.pkl', and 'label_encoder.pkl' exist.")
    exit()

# ----------------------------
# LOAD DATASET BARU
# ----------------------------
file_path = "/kaggle/input/test-akhir/hasil_preprocessing_test_whisper_sma.csv"
try:
    df_new = pd.read_csv(file_path)
    print(f"New dataset loaded successfully. Shape: {df_new.shape}")
except FileNotFoundError:
    print(f"File not found at {file_path}. Please check the path.")
    exit()

# Validasi kolom
if 'text_transcript' not in df_new.columns:
    text_col = df_new.select_dtypes(include=['object']).columns[0]
    df_new = df_new.rename(columns={text_col: 'text_transcript'})
    print(f"Renamed column to 'text_transcript'.")

# Pastikan kolom id ada
if 'id' not in df_new.columns:
    print("Error: Kolom 'id' tidak ditemukan dalam dataset.")
    exit()

# ----------------------------
# HANDLE MISSING VALUE DENGAN MEDIAN
# ----------------------------
# Hitung panjang kata untuk setiap teks
df_new['word_count'] = df_new['text_transcript'].apply(lambda x: len(str(x).split()) if pd.notna(x) else 0)

# Hitung median panjang kata (hanya untuk teks yang tidak kosong)
median_word_count = int(df_new[df_new['word_count'] > 0]['word_count'].median())
print(f"Median word count: {median_word_count}")

# Isi missing value dengan placeholder berdasarkan median
placeholder_text = "kosong " * median_word_count
df_new['text_transcript'] = df_new['text_transcript'].fillna(placeholder_text.strip())

# Drop kolom word_count karena sudah tidak diperlukan
df_new = df_new.drop(columns=['word_count'])

# ----------------------------
# PREPROCESSING DATA BARU
# ----------------------------
df_new['cleaned_text'] = df_new['text_transcript'].apply(clean_text)

# Jika cleaned_text kosong, isi dengan placeholder berdasarkan median
df_new['cleaned_text'] = df_new['cleaned_text'].apply(lambda x: placeholder_text.strip() if x == "" else x)

print(f"\nProcessed dataset: {len(df_new)} samples")
print("\nSample cleaned text:")
print(df_new[['id', 'text_transcript', 'cleaned_text']].head(3))

# ----------------------------
# TF-IDF VECTORIZATION
# ----------------------------
X_new_vec = vectorizer.transform(df_new['cleaned_text'])
print(f"\nTF-IDF New dataset shape: {X_new_vec.shape}")

# ----------------------------
# PREDIKSI
# ----------------------------
y_pred_new = model.predict(X_new_vec)
df_new['predicted_emotion'] = le.inverse_transform(y_pred_new)

# ----------------------------
# SORT DAN SIMPAN HASIL PREDIKSI
# ----------------------------
# Pilih hanya kolom id dan predicted_emotion, lalu urutkan berdasarkan id
output_df = df_new[['id', 'predicted_emotion']].sort_values(by='id')

# Simpan ke CSV
output_df.to_csv('predicted_emotions_bukti2.csv', index=False)
print("\nPredictions saved to 'predicted_emotions.csv'.")

Model, vectorizer, and encoder loaded successfully.
New dataset loaded successfully. Shape: (200, 4)
Median word count: 161

Processed dataset: 200 samples

Sample cleaned text:
    id                                    text_transcript  \
0    1  kali ini di sebelah saya sudah ada bike BG40 y...   
1   10  Ini mobil memang... ...saya saya kecaya enak k...   
2  100  Posis Sonipho Waduh, Jangan Start Back Start B...   

                                        cleaned_text  
0  kali sebelah bike road parking lihat, bemberny...  
1  mobil memang.. ..saya kecaya enak albota, emak...  
2  posis sonipho waduh, start back start block gi...  

TF-IDF New dataset shape: (200, 5000)

Predictions saved to 'predicted_emotions.csv'.


In [30]:
# ================================
# 8. BANDINGKAN DENGAN SUBMISSION LAMA
# ================================

submission_lama_path = "/kaggle/input/submit-waktu-itu/submissionSD2025040000155.csv"

try:
    # Load submission lama
    df_lama = pd.read_csv(submission_lama_path)
    print(f"\nSubmission lama loaded: {submission_lama_path}")
    print(f"Shape: {df_lama.shape}")

    # Pastikan kolom predicted ada
    if 'predicted' not in df_lama.columns:
        print("Error: Kolom 'predicted' tidak ditemukan di submission lama.")
        exit()

    # Pastikan id ada dan urutkan
    if 'id' not in df_lama.columns:
        df_lama['id'] = range(1, len(df_lama) + 1)
    df_lama = df_lama[['id', 'predicted']].sort_values('id').reset_index(drop=True)

    # Ambil hasil prediksi baru (yang sudah diurutkan)
    df_baru = output_df.copy()
    df_baru = df_baru.rename(columns={'predicted_emotion': 'predicted_baru'})
    df_baru = df_baru[['id', 'predicted_baru']].sort_values('id').reset_index(drop=True)

    # Validasi jumlah baris
    if len(df_lama) != len(df_baru):
        print(f"Warning: Jumlah baris tidak sama! Lama: {len(df_lama)}, Baru: {len(df_baru)}")
    else:
        print(f"Jumlah baris cocok: {len(df_lama)}")

    # Gabungkan untuk perbandingan
    df_compare = df_lama.copy()
    df_compare['predicted_baru'] = df_baru['predicted_baru'].values
    df_compare['berubah'] = df_compare['predicted'] != df_compare['predicted_baru']

    # Hitung statistik
    total_berubah = df_compare['berubah'].sum()
    persentase_berubah = (total_berubah / len(df_compare)) * 100

    print(f"\n{'='*50}")
    print(f" HASIL PERBANDINGAN PREDIKSI ")
    print(f"{'='*50}")
    print(f"Total baris       : {len(df_compare)}")
    print(f"Berubah           : {total_berubah} baris")
    print(f"Tetap sama        : {len(df_compare) - total_berubah} baris")
    print(f"Persentase berubah: {persentase_berubah:.2f}%")
    print(f"{'='*50}")

    # Mapping angka ke label untuk readability
    reverse_mapping = {v: k for k, v in mapping.items()}
    df_compare['label_lama'] = df_compare['predicted'].map(reverse_mapping)
    df_compare['label_baru'] = df_compare['predicted_baru'].map(reverse_mapping)

    # Tampilkan contoh perubahan
    df_berubah = df_compare[df_compare['berubah']]
    if len(df_berubah) > 0:
        print(f"\nContoh {min(15, len(df_berubah))} baris yang BERUBAH:")
        contoh = df_berubah[['id', 'label_lama', 'label_baru']].head(15)
        print(contoh.to_string(index=False))
    else:
        print("\nTidak ada perubahan! Semua prediksi sama.")

    # Simpan detail perbandingan
    diff_path = "diff_vs_submission_lama.csv"
    df_compare[['id', 'predicted', 'predicted_baru', 'label_lama', 'label_baru', 'berubah']].to_csv(diff_path, index=False)
    print(f"\nDetail perbandingan disimpan di: {diff_path}")

except FileNotFoundError:
    print(f"File tidak ditemukan: {submission_lama_path}")
    print("Pastikan path dan nama file benar.")
except Exception as e:
    print(f"Error saat membandingkan: {e}")


Submission lama loaded: /kaggle/input/submit-waktu-itu/submissionSD2025040000155.csv
Shape: (200, 2)
Jumlah baris cocok: 200

 HASIL PERBANDINGAN PREDIKSI 
Total baris       : 200
Berubah           : 200 baris
Tetap sama        : 0 baris
Persentase berubah: 100.00%

Contoh 15 baris yang BERUBAH:
 id label_lama label_baru
  1   Surprise        NaN
  2   Surprise        NaN
  3   Surprise        NaN
  4   Surprise        NaN
  5   Surprise        NaN
  6      Proud        NaN
  7   Surprise        NaN
  8   Surprise        NaN
  9   Surprise        NaN
 10   Surprise        NaN
 11   Surprise        NaN
 12        Joy        NaN
 13        Joy        NaN
 14   Surprise        NaN
 15   Surprise        NaN

Detail perbandingan disimpan di: diff_vs_submission_lama.csv


In [156]:
import pandas as pd
import re
import string
from io import StringIO
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, f1_score
import numpy as np
import joblib
import optuna

# Setup NLTK stopwords
try:
    nltk.download('stopwords', quiet=True)
    stop_words_id = set(stopwords.words('indonesian'))
    stop_words_en = set(stopwords.words('english'))
    stop_words = stop_words_id.union(stop_words_en)
except Exception as e:
    print("NLTK stopwords gagal di-download. Menggunakan manual list.")
    stop_words = {
        'dan', 'di', 'yang', 'untuk', 'dari', 'ke', 'pada', 'ini', 'itu', 
        'adalah', 'dengan', 'sebagai', 'oleh', 'akan', 'atau', 'tetapi', 
        'karena', 'jika', 'sementara', 'seperti', 'the', 'a', 'an', 'to', 
        'in', 'on', 'at', 'is', 'are', 'was', 'were', 'be', 'being', 'been'
    }

# Dictionary for typo/slang corrections (combined from train and test data)
typo_corrections = {
    'tau': 'tahu', 'ga': 'tidak', 'ngirip': 'menginap', 'gokil': 'gokil',
    'ngelakuin': 'melakukan', 'bember': 'bumper', 'fabricasi': 'fabrikasi',
    'kasteman': 'custom', 'spesia': 'spesial', 'kenal': 'kena', 'tong olan': 'tambahan',
    'foglem': 'fog lamp', 'menggauza': 'mengganti', 'kecaya': 'kece', 'emak': 'memang',
    'distirk': 'distrik', 'pretah': 'pret', 'kerasa': 'terasa', 'penghinder': 'penghindaran',
    'permasuk': 'memasukkan', 'kurosakan': 'kerusakan', 'komplikat': 'komplit',
    'denang': 'dengan', 'timboi': 'timbo', 'licin': 'licin', 'merundukan': 'merunduk',
    'kempad': 'kepad', 'stadia': 'stadi', 'tertar': 'tertarik', 'yui': 'yoi',
    'bernaval': 'bernavigasi', 'pukuran': 'pukulan', 'kula': 'bola', 'batel': 'badminton',
    'temok': 'tembok', 'memfaal': 'memfamiliarisasi', 'kancini': 'konsistensi',
    'cilatkan': 'latihkan', 'kongga': 'kanggo', 'tentar': 'tentu', 'berfit': 'fitbit',
    'latian': 'latihan', 'repetisi': 'repetisi', 'menunik': 'menu', 'cahat': 'catat',
    'hyper trophy': 'hypertrophy', 'beribadi': 'berbadan', 'apetama': 'pertama',
    'toba': 'coba', 'rupti': 'rupanya', 'dipsepak': 'disepak', 'berguru': 'belajar',
    'masal ap': 'muscle up', 'tepari': 'terapi', 'naret': 'narik', 'pudau': 'pull down',
    'injek': 'injak', 'pecep': 'cepat', 'spes': 'space', 'labor': 'elaborasi',
    'ngakat': 'mengangkat', 'pangga': 'panggil', 'arain': 'tarik', 'petalkan': 'otot kaki',
    'kamar': 'kamera', 'gaba': 'gambar', 'silaturah mi': 'silaturahmi', 'ganti-aki': 'ganti aki',
    'berpamitan': 'berpamitan', 'bodi wix': 'body weight', 'dital': 'ditarik',
    'glut beris': 'glute bridge', 'hanyok': 'hantuk', 'calf resist': 'calf raise',
    'tiba-tiba': 'tiba-tiba sekali', 'war oh': 'workout', 'lapan': 'lapangan',
    'OST-608': 'assisted pull-up', 'glandungan': 'gantungan', 'skap': 'scapular',
    'belau anang': 'berlawanan', 'unitoralis': 'universalis', 'enelurut': 'menurut',
    'dorasi': 'durasi', 'sumba': 'sumber', 'luma': 'lama', 'nakamru': 'makan',
    'dihidayah': 'dehidrasi', 'jaga': 'gagal', 'belifariasi': 'variasi', 'lesh': 'less',
    'sebesar': 'sebisa', 'hanyak': 'hanya', 'terlohor': 'telur', 'tempi': 'tempe',
    'makhlotis': 'makronutrien', 'balen': 'balik', 'panam': 'punggung', 'kesi imbakan': 'keseimbangan',
    'sita': 'sit-up', 'rancel': 'ransel', 'biasab': 'biceps', 'jatoh': 'jatuh',
    'sekolah': 'sekali', 'bantalan': 'bantalan bulat', 'romaine': 'roman', 'nge-strage': 'nge-stretch',
    'penunggam': 'punggung', 'ke egini': 'begini', 'bayset': 'biceps', 'gavis': 'gampang',
    'glamir-glamir': 'lemak-lemak', 'alara': 'olahraga', 'galain': 'gila', 'ngajem': 'nge-gym',
    'beringgui': 'beringas', 'richards': 'recharge', 'gok': 'gokil', 'mumpu': 'mumpuni',
    'kelampunya': 'lampunya', 'lay-bots': 'lay-ups', 'osmosan': 'sesak napas',
    'reen': 'ingin', 'jogin': 'jogging', 'jemerfuloh': 'sempurna', 'talis': 'tali',
    'patulari': 'patu lari', 'robong': 'lobang', 'mencakram': 'mengikat', 'perdam': 'perdam',
    'keling': 'keliling', 'peplafonnya': 'plafonnya', 'ngurangi': 'mengurangi', 'diikat': 'diikat',
    'otor s': 'otorisasi', 'koplingnya': 'kopling', 'tinggalkan': 'gunakan', 'bener-bener': 'benar-benar',
    'nyalan': 'nyalakan', 'in jap': 'injak', 'kematchetan': 'kecemasan', 'panyolis': 'panelis',
    'kentasiknya': 'fantastiknya', 'ketil': 'kecil', 'tampal': 'tampilan', 'bat-paks': 'batas',
    'plakson': 'klakson', 'kanyari-nyari': 'mencari-cari', 'jentat': 'gentar', 'hasar': 'hazard',
    'il as': 'hilang', 'pisa': 'pisah', 'peranya': 'pernah', 'kaunya': 'kamera', 'layak': 'layar',
    'seber-seberin': 'spek-spek', 'liras': 'lisensi', 'transyan': 'transision', 'grub': 'grup',
    'seteranya': 'storenya', 'kotus': 'kualitas', 'kukas': 'kulkas', 'memunuh': 'memenuhi',
    'killer': 'chiller', 'bawa-bahan': 'bahan-bahan', 'perluasan': 'ekspansi', 'peningin': 'pendingin',
    'di odorizer': 'deodorizer', 'sendap': 'bau', 'diffros': 'defrost', 'baska': 'basket',
    'pernahinannya': 'pemanasannya', 'buyang': 'buyar', 'halangan': 'haid', 'membalut': 'membelud',
    'lockmingsnya': 'locking-nya', 'menikwat': 'mengikat'
}

# Fungsi untuk membersihkan teks
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    # Hapus pola aneh
    text = re.sub(r'^(True|False)(,(True|False))*,,?', '', text)
    text = re.sub(r',,(True|False)(,(True|False))*$', '', text)
    # Hapus emoji dan non-ASCII
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    # Koreksi typo
    typo_corrections_local = typo_corrections.copy()
    for slang, formal in typo_corrections_local.items():
        pattern = r'\b' + re.escape(slang) + r'\b'
        text = re.sub(pattern, formal, text, flags=re.IGNORECASE)
    # Case folding
    text = text.lower()
    # Hapus URL, email, angka
    text = re.sub(r'http[s]?://\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'\d+', '', text)
    # Hapus tanda baca kecuali . ! ? ,
    keep_punct = ['!', '?', '.', ',']
    translator = str.maketrans('', '', ''.join([c for c in string.punctuation if c not in keep_punct]))
    text = text.translate(translator)
    # Kurangi huruf berulang
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    # Normalisasi spasi
    text = re.sub(r'\s+', ' ', text).strip()
    # Hapus stopwords
    words = [word for word in text.split() if word not in stop_words and len(word) > 2]
    text = ' '.join(words)
    return text.strip()

# Load dataset
file_path = "/kaggle/input/text-bdc/hasil_preprocessing.csv"
df = pd.read_csv(file_path)
print(f"File loaded successfully. Shape: {df.shape}")

# Validasi kolom
if 'text_transcript' not in df.columns:
    text_col = df.select_dtypes(include=['object']).columns[0]
    df = df.rename(columns={text_col: 'text_transcript'})
    print(f"Renamed column to 'text_transcript'.")

# Hapus baris dengan text_transcript kosong
df.dropna(subset=['text_transcript'], inplace=True)

# Encode label
le = LabelEncoder()
df['label'] = le.fit_transform(df['emotion'])
print(f"Classes: {le.classes_.tolist()}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    df['text_transcript'],
    df['label'],
    test_size=0.27,
    stratify=df['label'],
    random_state=42
)

# Buat DataFrame untuk train dan test
train_df = pd.DataFrame({'text': X_train, 'label': y_train})
test_df = pd.DataFrame({'text': X_test, 'label': y_test})

# Preprocessing teks
train_df['cleaned_text'] = train_df['text'].apply(clean_text)
test_df['cleaned_text'] = test_df['text'].apply(clean_text)

# Hapus baris dengan teks kosong setelah cleaning
train_df = train_df[train_df['cleaned_text'] != ""].reset_index(drop=True)
test_df = test_df[test_df['cleaned_text'] != ""].reset_index(drop=True)
print(f"\nTrain: {len(train_df)}, Test: {len(test_df)}")

# TF-IDF Vectorization
vectorizer = TfidfVectorizer(
    ngram_range=(1, 3),
    max_features=30000,
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    norm='l2',
    lowercase=False,
    stop_words=None
)

X_train_vec = vectorizer.fit_transform(train_df['cleaned_text'])
X_test_vec = vectorizer.transform(test_df['cleaned_text'])
print(f"TF-IDF Train shape: {X_train_vec.shape}")
print(f"TF-IDF Test shape: {X_test_vec.shape}")

# Modelling dengan Logistic Regression
best_C = 0.5010970952098729
best_penalty = 'l2'
model = LogisticRegression(
    C=best_C,
    penalty=best_penalty,
    solver='lbfgs',
    max_iter=5000,
    class_weight='balanced',
    random_state=42
)

# Latih model
model.fit(X_train_vec, train_df['label'])

# Prediksi pada data test
y_pred = model.predict(X_test_vec)

# Evaluasi model
print("\n=== 📊 EVALUASI MODEL (HYPERPARAMETER OPTUNA TRIAL #46) ===")
print("Confusion Matrix:")
print(confusion_matrix(test_df['label'], y_pred))
print("\nClassification Report:")
print(classification_report(test_df['label'], y_pred, target_names=le.classes_))
f1_macro = f1_score(test_df['label'], y_pred, average='macro')
print(f"\n🎯 F1-MACRO SCORE: {f1_macro:.5f}")

# Simpan model, vectorizer, dan label encoder
joblib.dump(model, 'logreg_optuna_trial46.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
joblib.dump(le, 'label_encoder.pkl')
print("\n✅ Model, vectorizer, dan label encoder berhasil disimpan!")

File loaded successfully. Shape: (738, 14)
Classes: ['Anger', 'Fear', 'Joy', 'Neutral', 'Proud', 'Sadness', 'Surprise', 'Trust']

Train: 538, Test: 200
TF-IDF Train shape: (538, 8207)
TF-IDF Test shape: (200, 8207)

=== 📊 EVALUASI MODEL (HYPERPARAMETER OPTUNA TRIAL #46) ===
Confusion Matrix:
[[ 6  0  0  0  2  0  2  0]
 [ 0  1  0  0  1  0  1  1]
 [ 0  0  3  0  2  0  6  3]
 [ 0  0  0  0  0  0  0  2]
 [ 0  0  6  0 10  0 16  7]
 [ 0  0  0  0  0  1  1  2]
 [ 3  0  4  0 13  4 45 14]
 [ 3  1  6  0 13  1  7 13]]

Classification Report:
              precision    recall  f1-score   support

       Anger       0.50      0.60      0.55        10
        Fear       0.50      0.25      0.33         4
         Joy       0.16      0.21      0.18        14
     Neutral       0.00      0.00      0.00         2
       Proud       0.24      0.26      0.25        39
     Sadness       0.17      0.25      0.20         4
    Surprise       0.58      0.54      0.56        83
       Trust       0.31      0.30

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



✅ Model, vectorizer, dan label encoder berhasil disimpan!


In [157]:
import pandas as pd
import joblib

# === 1. Load dataset baru ===
file_path_test = "/kaggle/input/test-text-bismillah/hasil_preprocessing_test_whisper_sma.csv"
df_new = pd.read_csv(file_path_test)
print(f"File test loaded. Shape: {df_new.shape}")

# Pastikan ada kolom teks
if 'text_transcript' not in df_new.columns:
    text_col = df_new.select_dtypes(include=['object']).columns[0]
    df_new = df_new.rename(columns={text_col: 'text_transcript'})
    print(f"Renamed column to 'text_transcript'.")

# === 2. Cleaning text (pakai fungsi clean_text dari code training sebelumnya) ===
df_new['cleaned_text'] = df_new['text_transcript'].apply(clean_text)

# Jangan buang row kosong → isi dengan "0"
df_new['cleaned_text'] = df_new['cleaned_text'].apply(lambda x: x if x != "" else "0")

# === 3. Load artefak training (model, vectorizer, encoder) ===
model = joblib.load("/kaggle/working/logreg_optuna_trial46.pkl")
vectorizer = joblib.load("/kaggle/working/tfidf_vectorizer.pkl")
le = joblib.load("/kaggle/working/label_encoder.pkl")

# === 4. Transform teks baru ke TF-IDF ===
X_new_vec = vectorizer.transform(df_new['cleaned_text'])

# === 5. Prediksi emotion ===
y_pred = model.predict(X_new_vec)
y_pred_label = le.inverse_transform(y_pred)

# === 6. Mapping label -> angka (sesuai permintaan) ===
mapping = {
    "Proud": 0,
    "Trust": 1,
    "Joy": 2,
    "Surprise": 3,
    "Neutral": 4,
    "Sadness": 5,
    "Fear": 6,
    "Anger": 7
}

# Buat dataframe hasil prediksi
df_result = pd.DataFrame({
    "id": df_new.index + 1 if "id" not in df_new.columns else df_new["id"],
    "predicted": [mapping.get(lbl, -1) for lbl in y_pred_label]  # -1 kalau label tidak ada di mapping
})

# Urutkan berdasarkan id
df_result = df_result.sort_values(by="id").reset_index(drop=True)

print("\n=== Hasil Prediksi (contoh 10 baris) ===")
print(df_result.head(10))

# === 7. Simpan hasil ke CSV (submission) ===
df_result.to_csv("submission.csv", index=False)
print("\n✅ File submission berhasil disimpan ke 'submission.csv'")

File test loaded. Shape: (200, 4)

=== Hasil Prediksi (contoh 10 baris) ===
   id  predicted
0   1          2
1   2          3
2   3          3
3   4          3
4   5          3
5   6          1
6   7          3
7   8          3
8   9          3
9  10          3

✅ File submission berhasil disimpan ke 'submission.csv'


In [158]:
# === 8. BANDINGKAN DENGAN SUBMISSION LAMA ===
submission_lama_path = "/kaggle/input/submit-waktu-itu/submissionSD2025040000155.csv"

try:
    # Load submission lama
    df_lama = pd.read_csv(submission_lama_path)
    print(f"\nSubmission lama loaded: {submission_lama_path}")
    print(f"Shape: {df_lama.shape}")

    # Pastikan kolom dan urutan id sama
    if 'id' not in df_lama.columns:
        df_lama['id'] = range(1, len(df_lama) + 1)
    df_lama = df_lama[['id', 'predicted']].sort_values('id').reset_index(drop=True)

    # Pastikan df_result juga sudah terurut
    df_baru = df_result.copy()
    df_baru = df_baru[['id', 'predicted']].sort_values('id').reset_index(drop=True)

    # Validasi jumlah baris
    if len(df_lama) != len(df_baru):
        print(f"Warning: Jumlah baris tidak sama! Lama: {len(df_lama)}, Baru: {len(df_baru)}")
    else:
        print(f"Jumlah baris cocok: {len(df_lama)}")

    # Hitung perbedaan
    df_compare = df_lama.copy()
    df_compare['predicted_baru'] = df_baru['predicted'].values
    df_compare['berubah'] = df_compare['predicted'] != df_compare['predicted_baru']

    total_berubah = df_compare['berubah'].sum()
    persentase_berubah = (total_berubah / len(df_compare)) * 100

    print(f"\n=== PERBANDINGAN HASIL ===")
    print(f"Total baris: {len(df_compare)}")
    print(f"Berubah: {total_berubah} baris")
    print(f"Tetap sama: {len(df_compare) - total_berubah} baris")
    print(f"Persentase berubah: {persentase_berubah:.2f}%")

    # Tampilkan contoh yang berubah (maks 10)
    df_berubah = df_compare[df_compare['berubah']]
    if len(df_berubah) > 0:
        print(f"\nContoh {min(10, len(df_berubah))} baris yang BERUBAH:")
        contoh = df_berubah[['id', 'predicted', 'predicted_baru']].head(10)
        # Konversi angka ke label untuk lebih mudah dibaca
        reverse_mapping = {v: k for k, v in mapping.items()}
        contoh['lama_label'] = contoh['predicted'].map(reverse_mapping)
        contoh['baru_label'] = contoh['predicted_baru'].map(reverse_mapping)
        print(contoh[['id', 'lama_label', 'baru_label']])
    else:
        print("\nTidak ada perubahan!")

    # Simpan diff (opsional)
    diff_path = "diff_vs_lama.csv"
    df_compare.to_csv(diff_path, index=False)
    print(f"\nDetail perbandingan disimpan di: {diff_path}")

except FileNotFoundError:
    print(f"File tidak ditemukan: {submission_lama_path}")
except Exception as e:
    print(f"Error saat membandingkan: {e}")


Submission lama loaded: /kaggle/input/submit-waktu-itu/submissionSD2025040000155.csv
Shape: (200, 2)
Jumlah baris cocok: 200

=== PERBANDINGAN HASIL ===
Total baris: 200
Berubah: 48 baris
Tetap sama: 152 baris
Persentase berubah: 24.00%

Contoh 10 baris yang BERUBAH:
    id lama_label baru_label
0    1   Surprise        Joy
5    6      Proud      Trust
11  12        Joy   Surprise
12  13        Joy   Surprise
17  18   Surprise      Trust
20  21      Proud      Trust
22  23    Neutral      Trust
26  27   Surprise      Trust
27  28      Trust      Proud
28  29   Surprise        Joy

Detail perbandingan disimpan di: diff_vs_lama.csv
